In [ ]:
%load_ext autoreload
%autoreload 2

pattern = "pattern-4"

entrypoint = pattern
app_cwl_file = f"../cwl-workflow/{pattern}.cwl"

try:
    from docs.helpers import plot_cwl, wrap_cwl
except (ImportError, ModuleNotFoundError) as e:

    import os
    import sys

    module_path = os.path.abspath(os.path.join("."))  # or the path to your source code
    sys.path.insert(0, module_path)

from helpers import WorkflowViewer, WorkflowWrapper
from cwl_loader import dump_cwl
from IPython.display import Markdown, display
import json

In [ ]:
wf = WorkflowViewer.from_file(app_cwl_file, entrypoint)

## Application Package Pattern 4

The CWL includes: 

- one input parameter of type `Directory`
- two output parameters of type `Directory`

This scenario takes as input an acquisition, applies two algorithms and generates two outputs.

Implementation: process the NDVI and NDWI taking as input a Landsat-8/9 acquisition. The output include two STAC Catalogs, each with a single STAC Item

In [ ]:
wf.plot()

### Inputs

In [ ]:
wf.display_inputs()

### Steps

In [ ]:
wf.display_steps()

### Outputs

In [ ]:
wf.display_outputs()

## Data flow management

In [ ]:
w = WorkflowWrapper(workflow=wf.workflow, entrypoint=entrypoint)
wrapped = w.wrap()

app_cwl_file = f".{entrypoint}.cwl"

with open(app_cwl_file, "w") as f:
    dump_cwl(process=wrapped, stream=f)

In [ ]:
wf = WorkflowViewer(cwl_file=app_cwl_file, workflow=wrapped, entrypoint="main")

wf.plot()

### Workflow sequence diagram

In [ ]:
wf.display_sequence_diagram()

### Inputs

In [ ]:
wf.display_inputs()

### Steps

In [ ]:
wf.display_steps()

### Outputs

In [ ]:
wf.display_outputs()

## Execution


In [ ]:
from cwltool.main import main
from io import StringIO
import argparse
import yaml

In [ ]:
params = {
    "aoi": "-118.985,38.432,-118.183,38.938",
    "epsg": "EPSG:4326",
    "item": {
        "class": "https://raw.githubusercontent.com/eoap/schemas/main/url.yaml#URL",
        "value": "https://planetarycomputer.microsoft.com/api/stac/v1/collections/landsat-c2-l2/items/LC08_L2SP_042033_20231007_02_T1",
    },
}
additional_params = {
    "another_input": "some_value",
    "s3_bucket": "my-bucket",
    "sub_path": "my/sub/path",
    "aws_access_key_id": "test",
    "aws_secret_access_key": "test",
    "region_name": "us-west-1",
    "endpoint_url": "https://s3.us-west-1.amazonaws.com",
}

with open(".params.yaml", "w") as f:
    yaml.dump({**params, **additional_params}, f)

md = f"""

### Inputs

```json
{json.dumps(params, indent=2)}
```
"""

display(Markdown(md))

In [ ]:
parsed_args = argparse.Namespace(
    podman=False,
    debug=False,
    validate=False,
    outdir="./runs",
    workflow=f"{app_cwl_file}#main",
    job_order=[".params.yaml"],
)

stream_out = StringIO()
stream_err = StringIO()

res = main(
    args=parsed_args,
    stdout=stream_out,
    stderr=stream_err,
)

if res == 0:
    md = f"""

### Outputs

```json
{stream_out.getvalue()}
```
    """
else:
    md = f"""

### Errors

CWL execution terminated with errors:

```
{stream_err.getvalue()}
```
    """

display(Markdown(md))